In [ ]:
!pip install biopython numpy meeko openbabel


In [1]:
from meeko import MoleculePreparation
from rdkit import Chem

# Load ligand from PDB
ligand_file = "remdesivir.pdb"
mol = Chem.MolFromPDBFile(ligand_file, removeHs=False)

# Prepare with Meeko
preparator = MoleculePreparation()
preparator.prepare(mol)

# Write PDBQT string
pdbqt_string = preparator.write_pdbqt_string()
print(pdbqt_string[:500])  # just show first 500 chars


A:\Anaconda\envs\docking_env\lib\site-packages\prody\utilities\misctools.py:4: DeprecationWarning: `np.chararray` is deprecated and will be removed from the main namespace in the future. Use an array with a string or bytes dtype instead.
  from numpy import unique, linalg, diag, sqrt, dot, chararray, divide, zeros_like, zeros, allclose, ceil, abs


REMARK SMILES CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@@](C#N)(c2ccc3c(N)ncnn23)[C@H](O)[C@@H]1O)Oc1ccccc1
REMARK SMILES IDX 12 1 13 3 14 4 15 5 16 6 17 7 18 8 34 9 19 10 32 11 35 12
REMARK SMILES IDX 22 14 31 15 23 16 30 17 25 18 24 19 29 20 26 21 28 22 27 23
REMARK SMILES IDX 20 26 21 27 33 28 36 30 37 31 38 32 42 33 39 34 41 35 40 36
REMARK SMILES IDX 10 37 11 38 8 39 9 40 7 41 6 42 3 43 2 44 1 45 4 46 5 47
REMARK H PARENT 12 2 35 13 27 24 27 25 33 29
ROOT
ATOM      1  N   UNL     1


A:\Anaconda\envs\docking_env\lib\site-packages\meeko\preparation.py:626: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
A:\Anaconda\envs\docking_env\lib\site-packages\meeko\preparation.py:420: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


In [8]:
with open("remdesiviriteration1.pdbqt", "w") as f:
    f.write(pdbqt_string)


In [6]:
!obabel 2FOM.pdb -O 2FOMiteration1.pdbqt -xr


*** Open Babel Warning  in OpenBabel::OBGlobalDataBase::Init
  Unable to open data file 'space-groups.txt'
*** Open Babel Warning  in OpenBabel::OBGlobalDataBase::Init
  Cannot initialize database 'space-groups.txt' which may cause further errors.
1 molecule converted


In [9]:
from Bio.PDB import PDBParser, NeighborSearch

# Load receptor PDB
parser = PDBParser(QUIET=True)
structure = parser.get_structure("receptor", "2FOM.pdb")

# Flatten all atoms
atoms = [atom for atom in structure.get_atoms()]


In [10]:
# List of residues in the binding site (example: Q-site)
binding_residues = ['GLY', 'SER', 'HIS', 'MET', 'LEU', 'GLU']  # adjust for your protein

# Collect all atoms in these residues
binding_atoms = [atom for atom in atoms if atom.get_parent().resname in binding_residues]

# Compute geometric center
x = sum(atom.coord[0] for atom in binding_atoms) / len(binding_atoms)
y = sum(atom.coord[1] for atom in binding_atoms) / len(binding_atoms)
z = sum(atom.coord[2] for atom in binding_atoms) / len(binding_atoms)

print("Center coordinates:", x, y, z)


Center coordinates: 2.4004147 -16.883986 14.888198


In [11]:
import numpy as np

coords = np.array([atom.coord for atom in binding_atoms])
min_coords = coords.min(axis=0)
max_coords = coords.max(axis=0)
margin = 4.0  # extra space around pocket

size_x, size_y, size_z = max_coords - min_coords + margin
print("Box size:", size_x, size_y, size_z)


Box size: 52.298 53.831997 46.319


In [12]:
config = f"""receptor = 2FOM.pdbqt
ligand = remdesivir.pdbqt

center_x = 2.40
center_y = -16.88
center_z = 14.89

size_x = 52.30
size_y = 53.83
size_z = 46.32

out = out.pdbqt
log = log.txt
"""

with open("configiteration1.txt", "w") as f:
    f.write(config)


In [14]:
!vina --receptor new2FOM.pdbqt --ligand remdesivir.pdbqt --config iteration1config.txt --out outiteration1.pdbqt --exhaustiveness 8


AutoDock Vina v1.2.7
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

In [1]:
# paste in a Jupyter cell to see volume and set a smaller box
size_x, size_y, size_z = 52.30, 53.83, 46.32
vol = size_x * size_y * size_z
print("Current search volume (Å^3):", vol)

# quick test: use ~30 Å in each dimension (adjust if ligand is big)
new_size_x, new_size_y, new_size_z = 30.0, 30.0, 30.0
print("Try box (Å):", new_size_x, new_size_y, new_size_z)


Current search volume (Å^3): 130405.11287999999
Try box (Å): 30.0 30.0 30.0


In [2]:
center_x, center_y, center_z = 2.40, -16.88, 14.89
config = f"""center_x = {center_x}
center_y = {center_y}
center_z = {center_z}

size_x = {new_size_x}
size_y = {new_size_y}
size_z = {new_size_z}
"""
with open("configiteration2.txt","w") as f:
    f.write(config)


In [7]:
!"C:\Users\Ayush\vina.exe" --receptor 2FOMiteration1.pdbqt --ligand remdesiviriteration1.pdbqt --config configiteration2.txt --out out_smallboxiteration2.pdbqt --exhaustiveness 8


AutoDock Vina v1.2.7
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V